# Web Scraping — KaBuM!

## Objetivo

Extrair **nome** e **preço** dos produtos de **Memória RAM** do site KaBuM.

## Imports

In [1]:
# %pip install requests
# %pip install bs4

import requests
from bs4 import BeautifulSoup
import json
import time
import random
import pathlib
import re
import pandas as pd

## Configurações

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36",
    "Accept-Language": "pt-BR,pt;q=0.9",
}

## Procurando Padrões

Procure na ultima linha: <script id="__NEXT_DATA__"... dentro desta div, temos a type="application/json", o ouro esta aqui

In [3]:
url = "https://www.kabum.com.br/hardware/memoria-ram"

response = requests.get(url, headers=headers)
print("Status:", response.status_code)

Status: 200


In [4]:
soup = BeautifulSoup(response.text, "html.parser")
soup

<!DOCTYPE html>
<html lang="pt-br"><head><link as="style" fetchpriority="high" href="https://static.kabum.com.br/conteudo/temas/001/header2.0/banner_monte_seu_pc.png" rel="preload"/><meta charset="utf-8"/><meta content="width=device-width, initial-scale=1.0" name="viewport"/><link href="https://images.kabum.com.br" rel="preconnect"/><link href="https://static.kabum.com.br" rel="preconnect"/><link href="https://themes.kabum.com.br" rel="preconnect"/><title>Memória RAM DDR5, DDR4 para PC e notebook</title><meta content="Memória RAM em promoção no KaBuM!
" name="title"/><meta content="Memórias RAM 8GB, 16GB e 32GB com descontos imperdíveis. Acelere seu PC ou notebook com performance confiável" name="description"/><link href="https://www.kabum.com.br/hardware/memoria-ram" rel="canonical"/><meta content="https://www.kabum.com.br/hardware/memoria-ram" property="og:url"/><meta content="website" property="og:type"/><meta content="Memória RAM em promoção no KaBuM!
" property="og:title"/><meta c

In [5]:
script_tag = soup.find("script", id="__NEXT_DATA__")
script_tag

<script id="__NEXT_DATA__" type="application/json">{"props":{"pageProps":{"data":"{\"catalogServer\":{\"meta\":{\"breadcrumb\":[{\"id\":1,\"name\":\"Hardware\",\"path\":\"/hardware\"},{\"id\":11,\"name\":\"Memória RAM\",\"path\":\"/memoria-ram\"}],\"seo\":{\"title\":\"Memória RAM DDR5, DDR4 para PC e notebook\",\"description\":\"Memórias RAM 8GB, 16GB e 32GB com descontos imperdíveis. Acelere seu PC ou notebook com performance confiável\",\"titleHeading\":\"Memória RAM em promoção no KaBuM!\\r\"},\"promotion\":{\"title\":\"\",\"description\":\"\"},\"totalItemsCount\":1151,\"totalPagesCount\":20,\"page\":{\"cursor\":\"\",\"number\":1,\"size\":60,\"isCurrentPage\":true},\"filters\":[{\"name\":\"has_flash\",\"values\":[false],\"text\":\"Entrega Flash\"},{\"name\":\"has_offer\",\"values\":[false,true],\"text\":\"Oferta\"},{\"name\":\"has_open_box\",\"values\":[false,true],\"text\":\"Openbox\"},{\"name\":\"has_prime\",\"values\":[false],\"text\":\"Prime\"},{\"name\":\"kabum_product\",\"valu

## Coleta de dados

In [7]:
next_data = json.loads(script_tag.string)
catalog = json.loads(next_data["props"]["pageProps"]["data"])

print(json.dumps(catalog["catalogServer"]["data"][0], indent=2, ensure_ascii=False))

{
  "code": 922165,
  "productSpecie": 0,
  "name": "Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4, CL22, Preto - HRM001083222PT",
  "description": "<h2>Mem&oacute;ria RAM Husky Impulse DDR4: Desempenho de Elite para o seu PC Gamer</h2>\n<p>A <strong>Mem&oacute;ria RAM Husky Impulse</strong> &eacute; o <strong>upgrade</strong> ideal para gamers que buscam a m&aacute;xima performance. Focada em velocidade e efici&ecirc;ncia, ela eleva a experi&ecirc;ncia de jogo a um novo patamar, garantindo que voc&ecirc; rode seus t&iacute;tulos favoritos com total fluidez, sem travamentos ou lentid&atilde;o.</p>\n<p>&nbsp;</p>\n<h3>Velocidade Extrema com 3200MHz</h3>\n<p>Prepare-se para uma jogabilidade mais suave e carregamentos mais r&aacute;pidos. Com sua frequ&ecirc;ncia de <strong>3200MHz</strong>, a <strong>Husky Impulse oferece</strong> um desempenho equilibrado e de alta velocidade. Essa caracter&iacute;stica &eacute; fundamental para jogos e aplicativos exigentes, permitindo que seu PC proce

localiza a tag __NEXT_DATA__ que o Next.js injeta no HTML, e faz dois json.loads porque pageProps["data"] é uma string JSON dentro do JSON principal.

In [16]:
# __NEXT_DATA__ — pageProps.data é uma string JSON, precisa parsear duas vezes
script_next = soup.find("script", id="__NEXT_DATA__")
next_data = json.loads(script_next.string)

# pageProps["data"] é uma string — precisa de um segundo json.loads
catalog = json.loads(next_data["props"]["pageProps"]["data"])

# print("Chaves do catalog:")
# print(list(catalog.keys()))

produtos_raw = catalog["catalogServer"]["data"]
print(f"\nProdutos encontrados: {len(produtos_raw)}")
# print("\nPrimeiro produto:")
print(produtos_raw[0])


Produtos encontrados: 60
{'code': 922165, 'productSpecie': 0, 'name': 'Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4, CL22, Preto - HRM001083222PT', 'description': '<h2>Mem&oacute;ria RAM Husky Impulse DDR4: Desempenho de Elite para o seu PC Gamer</h2>\n<p>A <strong>Mem&oacute;ria RAM Husky Impulse</strong> &eacute; o <strong>upgrade</strong> ideal para gamers que buscam a m&aacute;xima performance. Focada em velocidade e efici&ecirc;ncia, ela eleva a experi&ecirc;ncia de jogo a um novo patamar, garantindo que voc&ecirc; rode seus t&iacute;tulos favoritos com total fluidez, sem travamentos ou lentid&atilde;o.</p>\n<p>&nbsp;</p>\n<h3>Velocidade Extrema com 3200MHz</h3>\n<p>Prepare-se para uma jogabilidade mais suave e carregamentos mais r&aacute;pidos. Com sua frequ&ecirc;ncia de <strong>3200MHz</strong>, a <strong>Husky Impulse oferece</strong> um desempenho equilibrado e de alta velocidade. Essa caracter&iacute;stica &eacute; fundamental para jogos e aplicativos exigentes, permitindo

In [17]:
def extrair_produto(produto):
    return {
        "id":             produto.get("code"),
        "nome":           produto.get("name"),
        "preco_original": produto.get("oldPrice"),
        "preco_atual":    produto.get("price"),
        "preco_pix":      produto.get("priceWithDiscount"),
        "desconto_pct":   produto.get("discountPercentage"),
        "avaliacao":      produto.get("rating"),
        "num_avaliacoes": produto.get("ratingCount"),
        "disponivel":     produto.get("available"),
        "fabricante":     produto.get("manufacturer", {}).get("name"),
        "categoria":      produto.get("category"),
        "garantia":       produto.get("warranty"),
        "frete_gratis":   produto.get("flags", {}).get("isFreeShipping"),
        "url": f"https://www.kabum.com.br/produto/{produto.get('code')}/{produto.get('friendlyName', '')}",
    }

produtos_bs4 = [extrair_produto(p) for p in produtos_raw]
print(f"Produtos extraídos: {len(produtos_bs4)}")
produtos_bs4[0]

Produtos extraídos: 60


{'id': 922165,
 'nome': 'Memória RAM Husky Impulse, 8GB, 3200MHz, DDR4, CL22, Preto - HRM001083222PT',
 'preco_original': 0,
 'preco_atual': 858.81,
 'preco_pix': 729.99,
 'desconto_pct': 15,
 'avaliacao': 5,
 'num_avaliacoes': 180,
 'disponivel': True,
 'fabricante': 'Husky',
 'categoria': 'Hardware/Memória RAM/DDR 4/3200MHz',
 'garantia': '3 anos de garantia (3 meses de garantia legal + 33 meses de garantia contratual junto ao fabricante)',
 'frete_gratis': False,
 'url': 'https://www.kabum.com.br/produto/922165/memoria-ram-husky-impulse-8gb-3200mhz-ddr4-cl22-preto-hrm001083222pt'}

## Todas as páginas

In [19]:
todos_produtos = []

NUM_PAGINAS = 20  

for pagina in range(1, NUM_PAGINAS + 1):
    url = f"https://www.kabum.com.br/hardware/memoria-ram?page_number={pagina}"
    response = requests.get(url, headers=headers)
    print(f"Página {pagina}: {response.status_code}")

    soup = BeautifulSoup(response.text, "html.parser")
    script_next = soup.find("script", id="__NEXT_DATA__")
    catalog = json.loads(json.loads(script_next.string)["props"]["pageProps"]["data"])
    produtos_raw = catalog["catalogServer"]["data"]

    todos_produtos.extend([extrair_produto(p) for p in produtos_raw])
    time.sleep(random.uniform(0.5, 1.5))

print(f"\nTotal coletado: {len(todos_produtos)}")

Página 1: 200
Página 2: 200
Página 3: 200
Página 4: 200
Página 5: 200
Página 6: 200
Página 7: 200
Página 8: 200
Página 9: 200
Página 10: 200
Página 11: 200
Página 12: 200
Página 13: 200
Página 14: 200
Página 15: 200
Página 16: 200
Página 17: 200
Página 18: 200
Página 19: 200
Página 20: 200

Total coletado: 1151


## Salvando os dados

In [21]:
df = pd.DataFrame(todos_produtos)
# df.head()
# df.columns
df.shape

(1151, 14)

In [22]:
data_dir = pathlib.Path("data")
data_dir.mkdir(exist_ok=True)

# JSON
with (data_dir / "kabum_memoria_ram.json").open("w", encoding="utf-8") as f:
    json.dump(todos_produtos, f, ensure_ascii=False, indent=4)

# CSV
df = pd.DataFrame(todos_produtos)
df.to_csv(data_dir / "kabum_memoria_ram.csv", index=False, encoding="utf-8")

## Análises Futuras

Top 10 memórias mais baratas

In [ ]:
df.nsmallest(10, "preco_atual")[["nome", "preco_atual", "preco_pix", "desconto_pct"]]

,nome,preco_atual,preco_pix,desconto_pct
613,Memória P/ Notebook Ddr3 Pc3-10600 2gb,24.99,23.74,5
828,Memoria 1gb Ddr2 800mhz Sodimm,29.90,29.90,0
692,Memoria 2gb Ddr3 1333 Cl9 1.5v Desktop Sp312n...,54.90,51.06,7
1109,Memoria Kingston PC 1gb Ddr 400,55.77,55.77,0
445,"Memória Notebook Nanya, 2GB, 1333MHz, DDR3 - P...",60.01,60.01,0
582,"Memória Oxy, 4GB, 1600MHz, DDR3L, 1.35V, Para ...",62.63,62.63,0
557,"Memória Dale7, 2GB, 1333Mhz, DDR3, Desktop, 1.5v",63.89,63.89,0
863,"Memória Dale7, 2GB, 1600Mhz, DDR3, Desktop, 1.5v",63.89,63.89,0
606,"Memória Dale7, 2GB, 1333Mhz, DDR3, Notebook, 1.5v",67.89,67.89,0
740,"Memória Dale7, 2GB, 1600Mhz, DDR3, Notebook, 1.5v",67.89,67.89,0


Top 10 maiores descontos


In [25]:
df.nlargest(10, "desconto_pct")[["nome", "preco_original", "preco_atual", "desconto_pct"]]

,nome,preco_original,preco_atual,desconto_pct
192,"Memória Kingston DIMM, 8GB, 1600MHz, DDR3, 1.3...",0.0,824.0,22
218,"Memória Kingston HyperX Fury, 8GB, 3200MHz, DD...",0.0,849.3,22
292,"Memória Kingston SODIMM, 8GB, 2666MHz, DDR4, 1...",0.0,766.9,22
373,"Memória Kingston, 8GB, 1600MHz, DDR3, CL11, Pa...",0.0,758.0,22
513,"Memória Kingston, 4GB, 1600MHz, DDR3, CL11, Pa...",0.0,488.3,22
578,"Memória Kingston, 8GB, 2666MHz, DDR4, 1.2V, Pa...",0.0,744.3,22
1143,"Memória Kingston, 8GB, 1600MHz, DDR3, Para Not...",0.0,956.3,22
1144,"Memória Kingston, 4GB, 1600MHz, DDR3, Para Des...",0.0,544.3,22
1147,"Memória Kingston, 4GB, 1333MHz, DDR3, CL9",0.0,599.0,22
135,"Memória Gamer Xpg Spectrix D35g 16GB, RGB, DDR...",0.0,1305.0,20
